# 🚀 HUẤN LUYỆN VANILLA TRANSFORMER & ABLATION STUDY (ANH – VIỆT)
**Đề tài 8: Dịch máy Anh – Việt (IWSLT'15 En–Vi) | Môn học: Học sâu**  
**Phụ trách: Sơn (Phần 3 – Nhóm trưởng)**  
---
> **Lưu ý:** Chọn GPU trước khi chạy: **Runtime -> Change runtime type -> T4 GPU**.

### Bước 1: Kiểm tra GPU & Mount Google Drive chung

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone / Cập nhật mã nguồn từ GitHub & Cài thư viện

In [ ]:
import os

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
PROJECT_DIR = "/content/Machine_Translation_Project"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("✓ Đã sẵn sàng mã nguồn và thư viện!")

### Bước 3: Liên kết dữ liệu `data/processed/` & `custom_tokenizers/` từ Google Drive chung của nhóm

In [ ]:
# Cập nhật DRIVE_ROOT theo đúng đường dẫn thư mục Google Drive chung của Lê
DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"

!mkdir -p data/processed custom_tokenizers/word custom_tokenizers/bpe custom_tokenizers/spm checkpoints
!cp -r "{DRIVE_ROOT}/data/processed/." data/processed/
!cp -r "{DRIVE_ROOT}/custom_tokenizers/." custom_tokenizers/

!ls -lh data/processed/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Đăng nhập Weights & Biases (`btl-dl-translation-iwslt15`)

In [ ]:
import wandb
wandb.login()

### Bước 5: Huấn luyện Baseline Vanilla Transformer (Base: N=4, H=8, d_model=256, BPE)

In [ ]:
!python train_transformer.py \
    --config configs/transformer_config.yaml \
    --tokenizer bpe \
    --num_layers 4 \
    --num_heads 8 \
    --use_pe true \
    --smoothing 0.1 \
    --epochs 20 \
    --batch_size 128 \
    --save_dir checkpoints/transformer_bpe_base

### Bước 6: Đánh giá SacreBLEU trên tập Test `tst2013` & Sao lưu Checkpoint lên Drive

In [ ]:
# Greedy (beam_size=1) và Beam Search (beam_size=4)
!python eval.py --checkpoint checkpoints/transformer_bpe_base/best_model.pt --beam_size 1
!python eval.py --checkpoint checkpoints/transformer_bpe_base/best_model.pt --beam_size 4 --alpha 0.6

# Đồng bộ checkpoint & train_summary.json lên Google Drive chung
!mkdir -p "{DRIVE_ROOT}/checkpoints/transformer_bpe_base"
!cp -r checkpoints/transformer_bpe_base/. "{DRIVE_ROOT}/checkpoints/transformer_bpe_base/"

### Bước 7 (Giai đoạn 3): Thí nghiệm So sánh Tokenizer (Word / SPM) & Ablation Study (4 biến thể)

In [ ]:
# --- A. So sánh Tokenizer (Word & SentencePiece) ---
# !python train_transformer.py --tokenizer word --epochs 20 --save_dir checkpoints/transformer_word_base
# !python train_transformer.py --tokenizer spm  --epochs 20 --save_dir checkpoints/transformer_spm_base

# --- B. Ablation Study trên BPE Tokenizer ---
# Cấu hình 2 (Ablation số tầng: N=2, H=8)
# !python train_transformer.py --tokenizer bpe --num_layers 2 --num_heads 8 --epochs 20 --save_dir checkpoints/transformer_bpe_L2H8

# Cấu hình 3 (Ablation số head: N=4, H=4)
# !python train_transformer.py --tokenizer bpe --num_layers 4 --num_heads 4 --epochs 20 --save_dir checkpoints/transformer_bpe_L4H4

# Cấu hình 4 (Ablation tắt Positional Encoding: use_pe=False)
# !python train_transformer.py --tokenizer bpe --use_pe false --epochs 20 --save_dir checkpoints/transformer_bpe_noPE

# Cấu hình 5 (Ablation tắt Label Smoothing: smoothing=0.0)
# !python train_transformer.py --tokenizer bpe --smoothing 0.0 --epochs 20 --save_dir checkpoints/transformer_bpe_noLS